# Journey Planner — End-to-End Demo

This notebook demonstrates both IP formulations from the thesis:
1. **3D Matrix method** (Chapter 2) — synthetic examples
2. **TETN method** (Chapter 3) — real/synthetic Indian Railways data

In [ ]:
import sys
sys.path.insert(0, '..')

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

## Part 1: 3D Matrix Formulation Examples

In [ ]:
from src.formulation_3d import build_example_1, build_example_2, build_example_4, solve_3d
from src.verify import verify_3d_result, generate_verification_report

# Example 1: n=2, k=2
inp1 = build_example_1()
res1 = solve_3d(inp1)
print(f'Example 1 — Status: {res1.solver_status}, Objective: {res1.objective_value}')
print(f'  Tour: {res1.tour}')
print(f'  u-values: {res1.u_vals}')
print(f'  t-values: {res1.t_vals}')
print()

In [ ]:
# Example 4: n=4, k=3
inp4 = build_example_4()
res4 = solve_3d(inp4)
print(f'Example 4 — Status: {res4.solver_status}, Objective: {res4.objective_value}')
print(f'  Tour: {res4.tour}')
print(f'  u-values: {res4.u_vals}')
print(f'  t-values: {res4.t_vals}')

checks = verify_3d_result(inp4, res4)
report = generate_verification_report(checks)

## Part 2: Data Pipeline

In [ ]:
from src.data_pipeline import run_pipeline

schedule = run_pipeline()
print(f'Schedule: {len(schedule)} rows, {schedule["train_id"].nunique()} trains')
print(f'Stations: {schedule["station_name"].nunique()} unique')
schedule.head(10)

## Part 3: TETN Construction & Pruning

In [ ]:
from src.network import build_pruned_tetn, check_inter_destination_paths

stations = schedule['station_name'].unique()
destinations = list(stations[:3])
print(f'Query stations: {destinations}')

net, stats = build_pruned_tetn(schedule, destinations)
print(f'\nNetwork stats: {stats}')

paths = check_inter_destination_paths(net, destinations)
for p in paths:
    print(f"  {p['from']} -> {p['to']}: {'OK' if p['path_exists'] else 'MISSING'}")

## Part 4: TETN Solver (MDA3)

In [ ]:
from src.solver import solve_query, format_tour

origin = destinations[0]
intermediates = destinations[1:]

result = solve_query(
    origin=origin,
    destinations=intermediates,
    method='tetn_mda3',
    verify=True,
    schedule=schedule,
)

print(format_tour(result))